# Assignment 2 — MT2002 Statistical Modeling
**Semester:** Fall 2026  
**Deadline:** Tuesday, 20 October 2026, 11:50 PM

## General instructions
- Use the provided `tips.csv` dataset throughout. Do not replace or modify the source data.
- Complete every TODO, include the requested plots and interpretations, and run the notebook from top to bottom before submission.
- Give every plot a title and labeled axes. Keep code, summaries, diagnostics, and written interpretations visible in the submitted notebook.
- Choose priors appropriate to the scale of the outcome. Explain important modeling choices and interpret results in the restaurant context.
- Do not treat an extreme observation as an error solely because it is unusual. Do not interpret association as causation.

This assignment combines exploratory analysis, Bayesian regression, group comparisons, hierarchical modeling, posterior contrasts, and model checking.


The original assignment handout is included above. For this portfolio copy, the course CSV was unavailable, so the public reshape2 Tips CSV distributed through Rdatasets distributed through Rdatasets with the same schema and matching recorded summaries is bundled under data/. Original posterior estimates in the discussion refer to the submitted notebook’s saved run. Re-running can change Monte Carlo estimates and diagnostics.


## 1. Imports and shared setup
Import libraries for data handling, plotting, Bayesian modeling, posterior summaries, and probability calculations.

In [ ]:
from pathlib import Path
from IPython.display import display

import warnings
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="The figure layout has changed to tight")
az.style.use("arviz-darkgrid")
plt.rcParams["figure.dpi"] = 90
RNG_SEED = 42

DATA_PATH = next((path for path in (Path("data/tips.csv"), Path("../data/tips.csv"))
                  if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Open the notebook from the repository root or notebooks/.")

# Serial chains avoid Windows multiprocessing issues inside notebook kernels.
# Increase cores only when the local environment supports parallel sampling.
SAMPLE_KW = dict(draws=1000, tune=1000, chains=4, cores=1,
                 random_seed=RNG_SEED, progressbar=False)


def diagnostics(idata, name, *, exclude=()):
    """Print number of divergences and the worst r_hat / ESS for a fitted model."""
    variables = [name for name in idata.posterior.data_vars if name not in exclude]
    summ = az.summary(idata, var_names=variables, kind="diagnostics")
    n_div = int(idata.sample_stats["diverging"].sum())
    print(f"{name}: divergences = {n_div}, max r_hat = {summ['r_hat'].max():.3f}, "
          f"min ess_bulk = {summ['ess_bulk'].min():.0f}")


def trace(idata, var_names, title):
    az.plot_trace(idata, var_names=var_names, compact=True)
    plt.suptitle(title, y=1.02, fontsize=13)
    plt.show()


def ppc_plot(ppc, title, xlabel, ax=None):
    ax = az.plot_ppc(ppc, num_pp_samples=100, random_seed=RNG_SEED, ax=ax)
    ax = np.atleast_1d(ax)[0]
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    return ax


print("PyMC", pm.__version__, "| ArviZ", az.__version__)

## 2. Load and inspect the dataset
Load `tips.csv`, inspect its shape and columns, check missing values, and summarize numerical variables. Briefly describe any data-quality issues you identify.

In [ ]:
tips = pd.read_csv(DATA_PATH)

print("Shape:", tips.shape)
print("Columns:", list(tips.columns))
print("\nMissing values per column:")
print(tips.isna().sum())
print("\nDuplicated rows (ignoring the row id):", tips.drop(columns="rownames").duplicated().sum())
print("\nCategory levels:")
for col in ["sex", "smoker", "day", "time"]:
    print(f"  {col}: {tips[col].value_counts().to_dict()}")
print("\nBills per party size:", tips["size"].value_counts().sort_index().to_dict())
tips.describe()

**Data quality notes**

- There are 244 bills, 8 columns and no missing values. The types look right (numbers for bill, tip and size, text for the categories).
- One pair of rows (199 and 203) has exactly the same values: a $13.00 bill, a $2.00 tip, female smoker, Thursday lunch, party of 2. This could easily be two real tables, so I kept both.
- Party size is very unbalanced. Most bills are for 2 people (156), but only 4 bills are for 1 person, 5 for 5 people and 4 for 6 people. Estimates for those small groups will be uncertain.
- Bills and tips are right-skewed. The largest bill is $50.81 and the largest tip is $10. Many tips are round amounts (about 35% are whole dollars), so tips are not a smooth continuous variable.
- The data only covers one restaurant and 244 bills, so we can't generalise far beyond it.

I decided to keep the duplicated pair. A $13 bill with a $2 tip is a very normal order, and two separate tables at Thursday lunch could easily pay the same amount. Removing it would just be a guess, and the task also says not to change the data.

## 3. Exploratory analysis
Plot `total_bill` against `tip`. Describe the direction and apparent form of the relationship and identify observations that may be influential or unusual. Support your comments with the plot; do not delete observations just because they are extreme.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(tips["total_bill"], tips["tip"], alpha=0.6, label="Bills")
# mark the few largest tips / bills so they are easy to talk about
big = tips[(tips["tip"] >= 7) | (tips["total_bill"] >= 45)]
ax.scatter(big["total_bill"], big["tip"], facecolors="none", edgecolors="red", s=120,
           label="Large bill (≥ $45) or tip (≥ $7)")
ax.set_title("Tip amount vs total bill")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip ($)")
ax.legend()
plt.show()

big[["total_bill", "tip", "size", "sex", "smoker", "day", "time"]].sort_values("total_bill")

The relationship is positive and looks roughly linear: bigger bills usually come with bigger tips. The spread grows as the bill grows. Tips on $10 bills sit close together, while tips on $40–50 bills range from about $2.50 to $10. The circled points are the largest bills and tips (for example $50.81 with a $10 tip, and $48.33 with a $9 tip). Because they sit at the far right, they could pull the slope a lot. A few small bills also have unusually large tips, like a $7.25 bill with a $5.15 tip. These are real observations, so I keep them all and check their effect with a robust model in Section 5.

---
# PART 1 — Regression for Restaurant Operations
The restaurant operations team wants to understand which factors are associated with the **dollar amount of the tip** and to estimate tips for planning purposes.


## 4. Bayesian simple linear regression

Fit a Bayesian simple linear regression model

Use reasonable priors for the intercept, slope, and residual standard deviation. Sample from the posterior, inspect the parameter summary and trace plots, and visualize the posterior regression line with uncertainty.

**Interpretation tasks**
- Interpret the posterior for the slope in the context of the restaurant.
- Report the probability that the slope is positive.
- Explain what the uncertainty interval for the slope tells the manager.


In [ ]:
bill_raw = tips["total_bill"].values
tip_amt = tips["tip"].values

# Priors (dollar scale):
#  α ~ N(0, 2): tip for a $0 bill; should be near 0, a $2 sd still allows a few dollars
#  β ~ N(0, 0.5): extra tip per extra $1 of bill; 0.15-0.20 is usual, ±1 is very generous
#  σ ~ HalfNormal(2): typical scatter of tips around the line is a couple of dollars at most
with pm.Model() as tip_linear:
    α = pm.Normal("α", mu=0, sigma=2)
    β = pm.Normal("β", mu=0, sigma=0.5)
    σ = pm.HalfNormal("σ", sigma=2)
    μ = pm.Deterministic("μ", α + β * bill_raw)
    y = pm.Normal("y", mu=μ, sigma=σ, observed=tip_amt)
    idata_tip_linear = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_tip_linear, "Gaussian tip model")
display(az.summary(idata_tip_linear, var_names=["α", "β", "σ"]))

beta_tip = idata_tip_linear.posterior["β"].values.ravel()
print(f"P(β > 0) = {(beta_tip > 0).mean():.3f}")
print(f"Posterior mean of β = {beta_tip.mean():.3f}, 94% HDI = {az.hdi(beta_tip, hdi_prob=0.94)}")

trace(idata_tip_linear, ["α", "β", "σ"], "Trace plots: Gaussian regression of tip on total bill")

In [ ]:
order = np.argsort(bill_raw)
with tip_linear:
    ppc_tip_linear = pm.sample_posterior_predictive(idata_tip_linear, random_seed=RNG_SEED, progressbar=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bill_raw, tip_amt, alpha=0.5, label="Observed bills")
mu_mean = idata_tip_linear.posterior["μ"].mean(("chain", "draw")).values
ax.plot(bill_raw[order], mu_mean[order], color="k", lw=2, label="Posterior mean line")
az.plot_hdi(bill_raw, idata_tip_linear.posterior["μ"], hdi_prob=0.94, ax=ax,
            fill_kwargs={"alpha": 0.5, "label": "94% HDI of the mean"})
az.plot_hdi(bill_raw, ppc_tip_linear.posterior_predictive["y"], hdi_prob=0.94, ax=ax,
            fill_kwargs={"alpha": 0.15, "label": "94% HDI of predicted tips"})
ax.set_title("Bayesian simple linear regression: tip vs total bill")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip ($)")
ax.legend()
plt.show()

**Interpretation**

- **Priors:** α ~ N(0, 2), β ~ N(0, 0.5) and σ ~ HalfNormal(2), all in dollars. They allow any sensible tip pattern but rule out silly values, like the tip changing by $5 for every extra $1 of bill.
- **Slope:** the posterior mean of β is 0.105, with a 94% HDI of 0.092 to 0.119. So each extra $1 on the bill goes with about 10.5 cents more in tip. Put another way, a bill that is $10 higher comes with a tip about $1.05 higher on average.
- **P(β > 0) = 1.000.** Every one of the 4,000 posterior draws is positive, so the data leave no real doubt that the association is positive.
- **What the interval tells the manager:** the true extra tip per $10 of bill is very likely between about $0.92 and $1.19. That range is narrow enough to use for planning. But the band of predicted tips (light shading) is much wider, since σ ≈ $1.03. Single tables can still be far from the line.
- The sampler had no divergences, and r_hat = 1.00 for every parameter.

## 5. Robust regression: unusual bills or tips

**Scenario:** The manager is concerned that a small number of unusually large bills or tips may pull the fitted line away from the pattern followed by most customers.

Fit a robust regression model using a Student-t likelihood, keeping the predictor and response the same as in Section 4. Compare the slope and its uncertainty with the Gaussian model.

1. Present a plot comparing the two fitted relationships.
2. Explain whether the robust model changes the conclusion about the bill–tip relationship.
3. Do not claim that an observation is an error solely because it is extreme.


In [ ]:
# Same priors as the Gaussian model, plus ν (degrees of freedom).
# ν ~ Exponential(1/30) has mean 30, so it allows both heavy tails (small ν) and near-normal (large ν).
with pm.Model() as tip_robust:
    α = pm.Normal("α", mu=0, sigma=2)
    β = pm.Normal("β", mu=0, sigma=0.5)
    σ = pm.HalfNormal("σ", sigma=2)
    ν = pm.Exponential("ν", 1 / 30)
    μ = pm.Deterministic("μ", α + β * bill_raw)
    y = pm.StudentT("y", nu=ν, mu=μ, sigma=σ, observed=tip_amt)
    idata_tip_robust = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_tip_robust, "Student-t tip model")
display(az.summary(idata_tip_robust, var_names=["α", "β", "σ", "ν"]))
trace(idata_tip_robust, ["α", "β", "σ", "ν"], "Trace plots: robust (Student-t) regression of tip on total bill")

beta_rob = idata_tip_robust.posterior["β"].values.ravel()
comp = pd.DataFrame({
    "β mean": [beta_tip.mean(), beta_rob.mean()],
    "β sd": [beta_tip.std(), beta_rob.std()],
    "94% HDI low": [az.hdi(beta_tip)[0], az.hdi(beta_rob)[0]],
    "94% HDI high": [az.hdi(beta_tip)[1], az.hdi(beta_rob)[1]],
    "P(β > 0)": [(beta_tip > 0).mean(), (beta_rob > 0).mean()],
}, index=["Gaussian", "Student-t"])
display(comp.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bill_raw, tip_amt, alpha=0.4, color="grey", label="Observed bills")
for idata_, name, col in [(idata_tip_linear, "Gaussian", "C0"), (idata_tip_robust, "Student-t", "C1")]:
    m = idata_.posterior["μ"].mean(("chain", "draw")).values
    ax.plot(bill_raw[order], m[order], color=col, lw=2, label=f"{name} mean line")
    az.plot_hdi(bill_raw, idata_.posterior["μ"], hdi_prob=0.94, ax=ax, color=col,
                fill_kwargs={"alpha": 0.25, "label": f"{name} 94% HDI"})
ax.set_title("Gaussian vs Student-t regression of tip on total bill")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip ($)")
ax.legend()
plt.show()

**Robust vs Gaussian**

- The Student-t model gives β = 0.108 (94% HDI 0.093 to 0.123), compared with 0.105 (0.092 to 0.119) for the Gaussian model. The two fitted lines lie almost on top of each other in the plot.
- ν is about 4.6 (HDI 2.1 to 7.8). That is small, so the data have heavier tails than a normal distribution would allow. The robust model handles this by giving the points far from the line less weight, and its σ drops from 1.03 to 0.76.
- So the unusual bills and tips do **not** change the conclusion. Tips rise by about 10–11 cents for every extra dollar on the bill, and P(β > 0) is 1.000 in both models. The large points simply fit the general trend, so we have no reason to see them as errors.

## 6. Multivariable regression

**Scenario:** Bill amount alone may not explain differences in tips. The manager wants to consider party size as well.

Fit a model with `total_bill` and `size` as predictors:

\[
tip_i \sim \mathcal{N}(\mu_i,\sigma), \qquad
\mu_i = \alpha + \beta_1 total\_bill_i + \beta_2 size_i
\]

Use suitable priors and explain the interpretation of each slope while holding the other predictor constant. Compare the posterior estimates with the simple regression model. Comment on whether the additional predictor appears useful, based on posterior uncertainty and model checks rather than a single number alone.

In [ ]:
party = tips["size"].values

# β1 keeps the same prior as before; β2 ~ N(0, 1): one extra person changing the tip by more
# than about $2 (for the same bill) would be surprising.
with pm.Model() as tip_multi:
    α = pm.Normal("α", mu=0, sigma=2)
    β1 = pm.Normal("β1", mu=0, sigma=0.5)
    β2 = pm.Normal("β2", mu=0, sigma=1)
    σ = pm.HalfNormal("σ", sigma=2)
    μ = pm.Deterministic("μ", α + β1 * bill_raw + β2 * party)
    y = pm.Normal("y", mu=μ, sigma=σ, observed=tip_amt)
    idata_tip_multi = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_tip_multi, "Multivariable tip model")
display(az.summary(idata_tip_multi, var_names=["α", "β1", "β2", "σ"]))
trace(idata_tip_multi, ["α", "β1", "β2", "σ"], "Trace plots: tip ~ total bill + party size")

b1 = idata_tip_multi.posterior["β1"].values.ravel()
b2 = idata_tip_multi.posterior["β2"].values.ravel()
print(f"P(β1 > 0) = {(b1 > 0).mean():.3f},  P(β2 > 0) = {(b2 > 0).mean():.3f}")

# side-by-side comparison of the bill slope and residual sd
rows = {}
for name, idata_, bname in [("Simple", idata_tip_linear, "β"), ("Multivariable", idata_tip_multi, "β1")]:
    s = az.summary(idata_, var_names=[bname, "σ"])
    rows[name] = {"bill slope mean": s.loc[bname, "mean"], "bill slope 94% HDI": f"[{s.loc[bname, 'hdi_3%']}, {s.loc[bname, 'hdi_97%']}]",
                  "σ mean": s.loc["σ", "mean"]}
display(pd.DataFrame(rows).T)

# model checks: posterior predictive check and Bayesian R² (from μ draws)
with tip_multi:
    ppc_tip_multi = pm.sample_posterior_predictive(idata_tip_multi, random_seed=RNG_SEED, progressbar=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ppc_plot(ppc_tip_linear, "PPC: simple model (tip ~ bill)", "Tip ($)", ax=axes[0])
ppc_plot(ppc_tip_multi, "PPC: multivariable model (tip ~ bill + size)", "Tip ($)", ax=axes[1])
plt.tight_layout()
plt.show()

r2_rows = {}
for name, idata_ in [("Simple", idata_tip_linear), ("Multivariable", idata_tip_multi)]:
    mu_d = idata_.posterior["μ"].values.reshape(-1, len(tip_amt))
    r2_rows[name] = az.r2_score(tip_amt, mu_d)
display(pd.DataFrame(r2_rows).T.round(3))

fig, ax = plt.subplots(figsize=(7, 3))
az.plot_forest([idata_tip_linear, idata_tip_multi], model_names=["Simple (β)", "Multivariable (β1)"],
               var_names=["β", "β1"], combined=True, ax=ax)
ax.set_title("Bill slope in the simple and multivariable models (94% HDI)")
ax.set_xlabel("Extra tip ($) per $1 of bill")
plt.show()

**Interpretation**

- **β1 (bill)** = 0.093 (94% HDI 0.075 to 0.110). Comparing two tables with the **same party size**, the one with a $1 higher bill tips about 9 cents more.
- **β2 (size)** = 0.197 (94% HDI 0.050 to 0.361), with P(β2 > 0) = 0.993. Comparing two tables with the **same bill**, one extra person goes with a tip about 20 cents higher. The HDI is wide, though: the true value could be as low as 5 cents or as high as 36 cents.
- Compared with the simple model, the bill slope drops a little (0.105 → 0.093). This makes sense because bigger parties tend to have bigger bills, so the simple model gave part of the size effect to the bill.
- **Is size useful?** It helps a little, but not much. σ only moves from 1.027 to 1.017, and Bayesian R² goes from 0.456 to 0.468, which is well within one R² sd (0.035). The two posterior predictive checks look almost the same. Neither model captures the spikes at round tips ($2, $3), and both predict some impossible negative tips. So size has a credible but small association with the tip once bill is known. It barely improves prediction.

## 7. Short reflection — Part 1

In 3–5 sentences, compare the simple Gaussian, robust Student-t, and multivariable models. State what changed, what remained consistent, and one limitation of your analysis.

**Reflection: Part 1**

All three models agree that the tip rises with the bill, by about 9–11 cents per dollar. The Student-t model barely changed the slope (0.105 → 0.108) but found heavy tails (ν ≈ 4.6), so it gives the unusual tips less weight and has a smaller σ. Adding party size lowered the bill slope to 0.093 and gave a small positive size effect (≈ $0.20 per person), but the fit hardly improved (R² 0.456 → 0.468). One limitation is that every model uses a normal or Student-t likelihood on dollars, which allows negative tips and ignores the many round-number tips. Also, these are associations from one restaurant's bills, not causal effects.

If I had to give one model to the operations team, I would pick the simple Gaussian model with just the bill. It gives almost the same slope as the other two, it is the easiest to explain, and adding party size didn't really improve the fit.

---
# PART 2 — Tip-Rate Group Comparisons
## 2.1 Prepare the tip-rate outcome and groups
The owner is interested in how generously customers tip, not only the dollar amount. Define `tip_pct = 100 * tip / total_bill`. Compare tip rates by `smoker` and by a four-level `segment` combining `sex` and `smoker`. Fit Gaussian and Gamma likelihood models for both grouping variables.


In [ ]:
tips["tip_pct"] = 100 * tips["tip"] / tips["total_bill"]

smoker = tips["smoker"].astype("category")
smoker_idx = smoker.cat.codes.values
smoker_names = list(smoker.cat.categories)
n_smoker = len(smoker_names)
tip_pct = tips["tip_pct"].values
print("Smoker groups:", smoker_names)

tips.groupby("smoker")["tip_pct"].agg(["count", "mean", "median", "std", "min", "max"]).round(2)

## Tip rate by smoker Gaussian model

Model:
$
tip\_pct \sim \mathcal{N}(\mu_{smoker}, \sigma_{smoker})
$

- One mean per group: `mu_smoker[smoker_idx]`
- One `sigma` per group: `sigma[smoker_idx]`

In [ ]:
# Priors on the percentage scale: a mean tip rate of 16 ± 10 % covers anything plausible,
# and a within-group sd of up to ~20 percentage points is allowed.
with pm.Model(coords={"smoker": smoker_names}) as smoker_gauss:
    mu_smoker = pm.Normal("mu_smoker", mu=16, sigma=10, dims="smoker")
    sigma = pm.HalfNormal("sigma", sigma=10, dims="smoker")
    y = pm.Normal("y", mu=mu_smoker[smoker_idx], sigma=sigma[smoker_idx], observed=tip_pct)
    idata_smoker_gauss = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_smoker_gauss, "smoker_gauss")
display(az.summary(idata_smoker_gauss, var_names=["mu_smoker", "sigma"]))
trace(idata_smoker_gauss, ["mu_smoker", "sigma"], "Trace plots: tip rate by smoker (Gaussian)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
az.plot_forest(idata_smoker_gauss, var_names=["mu_smoker", "sigma"], combined=True, ax=axes[0])
axes[0].set_title("Smoker Gaussian model: group means and sds (94% HDI)")
axes[0].set_xlabel("Tip rate (%)")
with smoker_gauss:
    ppc_smoker_gauss = pm.sample_posterior_predictive(idata_smoker_gauss, random_seed=RNG_SEED, progressbar=False)
ppc_plot(ppc_smoker_gauss, "PPC: tip rate by smoker (Gaussian)", "Tip rate (%)", ax=axes[1])
plt.tight_layout()
plt.show()

The two group means are very close (15.93% for non-smokers vs 16.32% for smokers). But the smokers' spread is about twice as large (σ ≈ 8.6 vs 4.0). Because of that, the smokers' mean is less certain (posterior sd 0.88 vs 0.32).

## Tip rate by smoker  Gamma model

Tip rates are **positive** and **right-skewed** (a few tables tip a very large share of a small bill), so try a Gamma likelihood.

A simple parameterization is to model the **mean** per group with `mu_smoker > 0` and a shared `alpha` (shape):

$
tip\_pct \sim \text{Gamma}(\alpha, \beta = \alpha / \mu_{smoker})
$

In [ ]:
# mu_smoker > 0: HalfNormal(20) puts most mass below ~40 %.
# alpha (shape) ~ HalfNormal(20): a shape around 5-10 gives the right-skewed shape seen in the data.
with pm.Model(coords={"smoker": smoker_names}) as smoker_gamma:
    mu_smoker = pm.HalfNormal("mu_smoker", sigma=20, dims="smoker")
    alpha = pm.HalfNormal("alpha", sigma=20)

    y = pm.Gamma("y", alpha=alpha, beta=alpha / mu_smoker[smoker_idx], observed=tip_pct)
    idata_smoker_gamma = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_smoker_gamma, "smoker_gamma")
display(az.summary(idata_smoker_gamma, var_names=["mu_smoker", "alpha"]))
trace(idata_smoker_gamma, ["mu_smoker", "alpha"], "Trace plots: tip rate by smoker (Gamma, shared shape)")

# implied sd of each group under the Gamma model: sd = mu / sqrt(alpha)
post = idata_smoker_gamma.posterior
implied_sd = (post["mu_smoker"] / np.sqrt(post["alpha"])).mean(("chain", "draw")).values
gauss_s = az.summary(idata_smoker_gauss, var_names=["mu_smoker", "sigma"])
gamma_s = az.summary(idata_smoker_gamma, var_names=["mu_smoker"])
display(pd.DataFrame({
    "sample mean": tips.groupby("smoker")["tip_pct"].mean().values,
    "Gaussian mean": gauss_s["mean"].values[:2],
    "Gamma mean": gamma_s["mean"].values,
    "Gaussian mean sd": gauss_s["sd"].values[:2],
    "Gamma mean sd": gamma_s["sd"].values,
    "sample sd": tips.groupby("smoker")["tip_pct"].std().values,
    "Gaussian sigma": gauss_s["mean"].values[2:],
    "Gamma implied sd": implied_sd,
}, index=smoker_names).round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
az.plot_forest([idata_smoker_gauss, idata_smoker_gamma], model_names=["Gaussian", "Gamma"],
               var_names=["mu_smoker"], combined=True, ax=axes[0])
axes[0].set_title("Mean tip rate by smoker: Gaussian vs Gamma (94% HDI)")
axes[0].set_xlabel("Mean tip rate (%)")
with smoker_gamma:
    ppc_smoker_gamma = pm.sample_posterior_predictive(idata_smoker_gamma, random_seed=RNG_SEED, progressbar=False)
ppc_plot(ppc_smoker_gamma, "PPC: tip rate by smoker (Gamma)", "Tip rate (%)", ax=axes[1])
plt.tight_layout()
plt.show()

Under the Gamma model, the means hardly move (15.95% and 16.35%). The big difference is in the spread. With one shared shape α ≈ 8.5, the implied sd is about 5.5 for **both** groups. That is too wide for non-smokers (sample sd 3.99) and too narrow for smokers (sample sd 8.51).

## Tip rate by Customer Segment : Gaussian (group comparison)

Build a new grouping variable `segment` by combining `sex` and `smoker`, which gives four groups:
`Female-NonSmoker`, `Female-Smoker`, `Male-NonSmoker`, `Male-Smoker`. Model:

$
tip\_pct \sim \mathcal{N}(\mu_{segment}, \sigma_{segment})
$

Compare posterior means across the four `segment` groups.

In [ ]:
# Build the segment column
tips["segment"] = tips["sex"] + "-" + tips["smoker"].map({"Yes": "Smoker", "No": "NonSmoker"})

# Encode segment groups
segments = np.sort(tips["segment"].unique())
seg_idx = pd.Categorical(tips["segment"], categories=segments).codes
n_seg = len(segments)
display(tips.groupby("segment")["tip_pct"].agg(["count", "mean", "std"]).round(2))

# Same weakly informative priors as the smoker Gaussian model
with pm.Model(coords={"segment": segments}) as segment_gauss:
    mu_seg = pm.Normal("mu_seg", mu=16, sigma=10, dims="segment")
    sigma = pm.HalfNormal("sigma", sigma=10, dims="segment")
    y = pm.Normal("y", mu=mu_seg[seg_idx], sigma=sigma[seg_idx], observed=tip_pct)
    idata_segment_gauss = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_segment_gauss, "segment_gauss")
# 1) Posterior summary for mu_seg
display(az.summary(idata_segment_gauss, var_names=["mu_seg", "sigma"]))
trace(idata_segment_gauss, ["mu_seg", "sigma"], "Trace plots: tip rate by segment (Gaussian)")

# 2) Posterior distribution of mean tip rate for each segment
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
az.plot_forest(idata_segment_gauss, var_names=["mu_seg"], combined=True, ax=axes[0])
axes[0].set_title("Mean tip rate by segment, Gaussian model (94% HDI)")
axes[0].set_xlabel("Mean tip rate (%)")

# 3) Posterior predictive check
with segment_gauss:
    ppc_segment_gauss = pm.sample_posterior_predictive(idata_segment_gauss, random_seed=RNG_SEED, progressbar=False)
ppc_plot(ppc_segment_gauss, "PPC: tip rate by segment (Gaussian)", "Tip rate (%)", ax=axes[1])
plt.tight_layout()
plt.show()

## Tip rate by Customer Segment Gamma (group comparison)

Because `tip_pct` is positive, a Gamma likelihood is often appropriate. This time use PyMC's mean / standard-deviation parameterization, with one mean **and** one standard deviation per segment:

$
tip\_pct \sim \text{Gamma}(\mu = \mu_{segment},\ \sigma = \sigma_{segment})
$

Compare posterior means across segments and contrast with the Gaussian model.

In [ ]:
# Means must be positive -> HalfNormal(20); sds HalfNormal(10) as before.
with pm.Model(coords={"segment": segments}) as segment_gamma:
    mu_seg = pm.HalfNormal("mu_seg", sigma=20, dims="segment")
    sigma = pm.HalfNormal("sigma", sigma=10, dims="segment")
    y = pm.Gamma("y", mu=mu_seg[seg_idx], sigma=sigma[seg_idx], observed=tip_pct)
    idata_segment_gamma = pm.sample(**SAMPLE_KW)

diagnostics(idata_segment_gamma, "segment_gamma")
display(az.summary(idata_segment_gamma, var_names=["mu_seg", "sigma"]))
trace(idata_segment_gamma, ["mu_seg", "sigma"], "Trace plots: tip rate by segment (Gamma)")

fig, ax = plt.subplots(figsize=(7, 4))
az.plot_forest(idata_segment_gamma, var_names=["mu_seg"], combined=True, ax=ax)
ax.set_title("Mean tip rate by segment, Gamma model (94% HDI)")
ax.set_xlabel("Mean tip rate (%)")
plt.show()

In [ ]:
g = az.summary(idata_segment_gauss, var_names=["mu_seg"])
m = az.summary(idata_segment_gamma, var_names=["mu_seg"])
seg_table = pd.DataFrame({
    "n": tips.groupby("segment")["tip_pct"].count().values,
    "sample mean": tips.groupby("segment")["tip_pct"].mean().values,
    "Gaussian mean": g["mean"].values,
    "Gaussian 94% HDI": [f"[{a}, {b}]" for a, b in zip(g["hdi_3%"], g["hdi_97%"])],
    "Gamma mean": m["mean"].values,
    "Gamma 94% HDI": [f"[{a}, {b}]" for a, b in zip(m["hdi_3%"], m["hdi_97%"])],
}, index=segments)
display(seg_table.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
az.plot_forest([idata_segment_gauss, idata_segment_gamma], model_names=["Gaussian", "Gamma"],
               var_names=["mu_seg"], combined=True, ax=ax)
ax.set_title("Mean tip rate by segment: Gaussian vs Gamma (94% HDI)")
ax.set_xlabel("Mean tip rate (%)")
plt.show()

with segment_gamma:
    ppc_segment_gamma = pm.sample_posterior_predictive(idata_segment_gamma, random_seed=RNG_SEED, progressbar=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ppc_plot(ppc_segment_gauss, "PPC: segment Gaussian", "Tip rate (%)", ax=axes[0])
ppc_plot(ppc_segment_gamma, "PPC: segment Gamma", "Tip rate (%)", ax=axes[1])
for a in axes:
    a.set_xlim(-10, 75)
plt.tight_layout()
plt.show()

# How much predicted mass falls below 0 % (impossible values)?
neg_share = (ppc_segment_gauss.posterior_predictive["y"] < 0).mean().item()
print(f"Share of Gaussian predicted tip rates below 0%: {neg_share:.3%}")
print(f"Share of Gamma predicted tip rates below 0%: {(ppc_segment_gamma.posterior_predictive['y'] < 0).mean().item():.3%}")

**Gaussian vs Gamma for segments**

- The two models rank the segments the same way and give very similar means. Female smokers come out highest (18.18% Gaussian, 18.39% Gamma), and the other three sit between about 15.3% and 16.1%.
- The biggest difference is for Male-Smoker. This group holds the 71% tip rate. In the Gaussian model that one point pushes σ up to 9.18, and the mean's HDI is 13.04 to 17.53. The Gamma model already expects a long right tail, so its σ is 7.33 and its HDI is narrower (13.64 to 17.20).
- In the posterior predictive checks, the Gaussian model puts about 1.35% of predicted tip rates below 0%, which is impossible. The Gamma model never does, and it follows the skewed shape of the data a bit better. Both models still have trouble reaching the single 71% value.

----

## Short reflection (2–4 sentences on PART 1)
Smokers and non-smokers have almost the same average tip rate but a very different spread. Explain how and why the two likelihoods (Gaussian with one `sigma` per group vs. Gamma with a shared `alpha`) handle this differently, and how that impacts the estimation of the group means.

**Reflection**

In the Gaussian model, each group gets its own σ. So the smokers' large spread (σ ≈ 8.6 vs 4.0) is kept separate, and only the smokers' mean becomes uncertain (posterior sd 0.88 vs 0.32 for non-smokers). The Gamma model with one shared α forces the same relative spread on both groups (sd = μ/√α ≈ 5.5 for both). It over-states the noise for non-smokers and under-states it for smokers. As a result, the Gamma model is less sure about the non-smoker mean (sd 0.45) and more sure about the smoker mean (sd 0.58) than it should be. The point estimates of the means barely change (≈ 15.9% and 16.3%), but the uncertainty is shared out wrongly between the groups, and that matters when we compare them.

## Posterior Interpretation

Using the **Gaussian segment model**, compute the pairwise posterior differences between the group means for all groups (4 segments, so 6 comparisons). For each comparison, calculate Cohen's d and the Probability of Superiority. Plot the posterior distributions of the mean differences, similar to the lecture notebook, and briefly interpret what the graphs indicate about the differences between the groups.

Then do the same for the single **Smoker vs Non-Smoker** comparison from the Gaussian smoker model.

In your interpretation, answer: does any segment stand out, and is the evidence strong enough to say so with confidence?

In [ ]:
def compare_pair(mu_a, mu_b, sd_a, sd_b):
    """Posterior difference, Cohen's d and probability of superiority for two groups."""
    diff = mu_a - mu_b
    d = diff / np.sqrt((sd_a**2 + sd_b**2) / 2)
    ps = stats.norm.cdf(d / np.sqrt(2))
    hdi = az.hdi(diff, hdi_prob=0.94)
    return diff, {"mean diff (pp)": diff.mean(), "HDI low": hdi[0], "HDI high": hdi[1],
                  "P(diff > 0)": (diff > 0).mean(), "Cohen's d": d.mean(),
                  "Prob. of superiority": ps.mean()}

post_seg = az.extract(idata_segment_gauss)
mu_s = post_seg["mu_seg"].values    # shape (segment, sample)
sd_s = post_seg["sigma"].values

pairs = list(itertools.combinations(range(n_seg), 2))
rows, diffs = {}, {}
for i, j in pairs:
    label = f"{segments[i]} − {segments[j]}"
    diffs[label], rows[label] = compare_pair(mu_s[i], mu_s[j], sd_s[i], sd_s[j])
pair_table = pd.DataFrame(rows).T
display(pair_table.round(3))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (label, diff) in zip(axes.ravel(), diffs.items()):
    az.plot_posterior(diff, ref_val=0, hdi_prob=0.94, ax=ax)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Difference in mean tip rate (pp)")
fig.suptitle("Pairwise posterior differences between segment means (Gaussian segment model)", fontsize=13)
plt.tight_layout()
plt.show()

# Smoker vs Non-smoker from the Gaussian smoker model
post_sm = az.extract(idata_smoker_gauss)
i_yes, i_no = smoker_names.index("Yes"), smoker_names.index("No")
diff_sm, row_sm = compare_pair(post_sm["mu_smoker"].values[i_yes], post_sm["mu_smoker"].values[i_no],
                               post_sm["sigma"].values[i_yes], post_sm["sigma"].values[i_no])
display(pd.DataFrame(row_sm, index=["Smoker − NonSmoker"]).round(3))

fig, ax = plt.subplots(figsize=(7, 4))
az.plot_posterior(diff_sm, ref_val=0, hdi_prob=0.94, ax=ax)
ax.set_title("Smoker − NonSmoker: difference in mean tip rate (Gaussian smoker model)")
ax.set_xlabel("Difference in mean tip rate (pp)")
plt.show()

**Interpretation**

- Female smokers are the only segment that looks different. They tip about 2.1 pp more than male non-smokers (P = 0.945), 2.9 pp more than male smokers (P = 0.955) and 2.5 pp more than female non-smokers (P = 0.966). The other three comparisons are centred near zero (differences of 0.4–0.8 pp, P between 0.29 and 0.73).
- Even so, the evidence isn't strong. All six 94% HDIs include 0. Cohen's d for the female-smoker comparisons is only about 0.35–0.43 (a small-to-medium effect). The probability of superiority is about 0.60, meaning a random female smoker out-tips a random customer from another segment only about 60% of the time. Female smokers are also a small group (33 bills) with a wide spread (σ ≈ 7.4).
- Smokers vs non-smokers: the difference is 0.39 pp (HDI −1.33 to 2.17), P = 0.663, d = 0.06 and PS = 0.52. This is basically no difference in average tip rate.
- So female smokers *might* tip a slightly higher share, but I wouldn't say this with confidence from 244 bills. Being a smoker or a woman also doesn't cause the higher tip rate. Other things, like the day or the time of the meal, could be involved.

In practice I don't think a 2–3 point gap matters much. On a $20 bill that is only about 40–60 cents, and since the evidence isn't even strong, I wouldn't change service or staffing based on it.

***

---
# PART 3 — Hierarchical Modeling by Party Size
Create a hierarchical version of the tip-rate example, partially pooling across party sizes (`size` = 1, 2, ..., 6). Compare independent group estimates with hierarchical estimates. First inspect how many bills are available for each party size.


Create a hierarchical version of the tip-rate example, by partially pooling across the **party sizes** (`size` = 1, 2, ..., 6). Compare the results to those obtained without the hierarchical structure.

The party sizes are very unbalanced, so before modelling check how many bills there are for each size.

First we are going to load the data and refit the model with gamma distribution.

In [ ]:
tips = pd.read_csv(DATA_PATH)
tips["tip_pct"] = 100 * tips["tip"] / tips["total_bill"]
tips.groupby("size")["tip_pct"].agg(["count", "mean", "std"]).round(2)

In [ ]:
categories = np.array([1, 2, 3, 4, 5, 6])

tip_pct = tips["tip_pct"].values
idx = pd.Categorical(tips["size"], categories=categories).codes

In [ ]:
coords = {"size": categories}

# μ ~ HalfNormal(20): positive mean tip rate, most mass below ~40 %.
# σ ~ HalfNormal(10): within-size spread of up to about 20 percentage points.
with pm.Model(coords=coords) as comparing_groups_nh:
    μ = pm.HalfNormal("μ", sigma=20, dims="size")
    σ = pm.HalfNormal("σ", sigma=10, dims="size")
    y = pm.Gamma("y", mu=μ[idx], sigma=σ[idx], observed=tip_pct)
    idata_cg_nh = pm.sample(**SAMPLE_KW)

diagnostics(idata_cg_nh, "non-hierarchical")
display(az.summary(idata_cg_nh, var_names=["μ", "σ"]))
trace(idata_cg_nh, ["μ", "σ"], "Trace plots: non-hierarchical Gamma model by party size")

In [ ]:
# Hyperprior for the overall mean tip rate: Gamma with mean 16 % and sd 5 %.
with pm.Model(coords=coords) as comparing_groups_h_00:
    μ_g = pm.Gamma("μ_g", mu=16, sigma=5)
    μ = pm.Gamma("μ", mu=μ_g, sigma=1, dims="size")        # between-group spread fixed at 1 pp
    σ = pm.HalfNormal("σ", sigma=10, dims="size")
    y = pm.Gamma("y", mu=μ[idx], sigma=σ[idx], observed=tip_pct)
    idata_cg_h_00 = pm.sample(**SAMPLE_KW)

diagnostics(idata_cg_h_00, "hierarchical_00 (fixed spread)")
display(az.summary(idata_cg_h_00, var_names=["μ_g", "μ", "σ"]))
trace(idata_cg_h_00, ["μ_g", "μ", "σ"], "Trace plots: hierarchical_00 (between-group spread fixed at 1)")

# σ_g ~ HalfNormal(5): party-size means are expected to differ by a few points, not by tens.
with pm.Model(coords=coords) as comparing_groups_h_01:
    μ_g = pm.Gamma("μ_g", mu=16, sigma=5)
    σ_g = pm.HalfNormal("σ_g", sigma=5)
    μ = pm.Gamma("μ", mu=μ_g, sigma=σ_g, dims="size")
    σ = pm.HalfNormal("σ", sigma=10, dims="size")
    y = pm.Gamma("y", mu=μ[idx], sigma=σ[idx], observed=tip_pct)
    idata_cg_h_01 = pm.sample(**SAMPLE_KW, target_accept=0.99)

diagnostics(idata_cg_h_01, "hierarchical_01 (learned spread)")
display(az.summary(idata_cg_h_01, var_names=["μ_g", "σ_g", "μ", "σ"]))
trace(idata_cg_h_01, ["μ_g", "σ_g", "μ", "σ"], "Trace plots: hierarchical_01 (between-group spread learned)")

# Where do the divergences happen? Compare σ_g in divergent vs normal draws.
div = idata_cg_h_01.sample_stats["diverging"].values.ravel()
sg = idata_cg_h_01.posterior["σ_g"].values.ravel()
if div.any():
    print(f"σ_g in divergent draws: mean {sg[div].mean():.2f} | all draws: mean {sg.mean():.2f}")
else:
    print("No divergent draws in this run.")
ax = az.plot_pair(idata_cg_h_01, var_names=["σ_g", "μ"], coords={"size": [1]}, divergences=True,
                  scatter_kwargs={"alpha": 0.3})
ax.set_title("hierarchical_01: σ_g vs μ[1] (divergent draws marked)")
ax.set_xlabel("σ_g (between-size spread, pp)")
ax.set_ylabel("μ[1] (mean tip rate, size 1, %)")
plt.show()

## Forest Plot

In [ ]:
axes = az.plot_forest([idata_cg_nh, idata_cg_h_00, idata_cg_h_01],
                      model_names=['non_hierarchical', 'hierarchical_00', 'hierarchical_01'],
                      var_names=['μ'], combined=True, r_hat=False, ess=False, figsize=(12, 5),
                      colors='cycle')
y_lims = axes[0].get_ylim()
axes[0].vlines(idata_cg_h_01.posterior['μ_g'].mean(), *y_lims, color="k", ls=":");

*The x-axis of the forest plot is the mean tip rate (%) for each party size. The dotted line is the posterior mean of the overall mean μ_g from hierarchical_01. This cell was provided, so I left it unchanged.*

## Shrinkage Table
Build one table with a row for each party size and these columns: number of bills, sample mean of `tip_pct`, and the posterior mean of `μ` from each of the three models.

In [ ]:
grp = tips.groupby("size")["tip_pct"]
shrink = pd.DataFrame({
    "n bills": grp.count().reindex(categories).values,
    "sample mean": grp.mean().reindex(categories).values,
    "non_hierarchical μ": idata_cg_nh.posterior["μ"].mean(("chain", "draw")).values,
    "hierarchical_00 μ": idata_cg_h_00.posterior["μ"].mean(("chain", "draw")).values,
    "hierarchical_01 μ": idata_cg_h_01.posterior["μ"].mean(("chain", "draw")).values,
}, index=pd.Index(categories, name="size"))
shrink["shift h_01 vs sample"] = shrink["hierarchical_01 μ"] - shrink["sample mean"]
print(f"Posterior mean of μ_g (hierarchical_01): {idata_cg_h_01.posterior['μ_g'].mean().item():.2f}")
print(f"Posterior mean of σ_g (hierarchical_01): {idata_cg_h_01.posterior['σ_g'].mean().item():.2f}")
shrink.round(2)

## Short reflection (2–4 sentences on PART 2)
Explain how the gamma distribution as likelihood impacts the model, compare the models from the forest plot, and explain the shrinkage effect. Which party sizes are shrunk the most, in which direction, and why? What changes between `hierarchical_00` (fixed spread between groups) and `hierarchical_01` (spread learned from the data)?

**Reflection**

The Gamma likelihood keeps every tip rate positive and allows a long right tail. Each size also has its own σ, so the very variable small groups (sizes 1 and 5) get wide intervals instead of affecting the other groups. In the forest plot, the non-hierarchical model gives very wide intervals for sizes 1, 5 and 6 (for size 1, 14.3 to 30.0). Both hierarchical models pull those estimates towards the overall mean (μ_g ≈ 16) and make the intervals much narrower. Size 1 is shrunk the most, from a sample mean of 21.73 down to 16.90 (h_01). Size 5 is pulled up from 14.15 to 15.69. Size 2, with 156 bills, barely moves (−0.11). This happens because groups with only 4–5 bills carry little information, so the shared distribution matters more for them. With the spread fixed at 1 pp, `hierarchical_00` shrinks the hardest (size 1 → 16.20). `hierarchical_01` learns σ_g ≈ 1.5 (HDI 0.07 to 3.47), so it shrinks a bit less and gives wider, more honest intervals. It also had 9 divergences, all where σ_g is near zero (mean σ_g 0.17 in divergent draws). With r_hat ≤ 1.01 the results look usable, but its σ_g estimate (ess_bulk 342) should be treated with some care.

---
# PART 4 — Regression for Tip Rate
Investigate whether customers tip a smaller **percentage** of the bill as the bill gets larger. Fit Gaussian linear, robust Student-t, quadratic, robust quadratic, and multivariable models. Centre `total_bill` before modeling so that the intercept represents the expected tip rate for an average bill.


# Tips dataset (Regression: tip rate vs bill amount)

Do customers tip a smaller share when the bill gets larger? Model `tip_pct` as a function of `total_bill`.

You will fit **four Bayesian regression models**:
1) **Simple linear** regression with a Gaussian likelihood
2) **Robust linear** regression with a Student-t likelihood
3) **Polynomial** (quadratic) regression
4) **Multivariable** regression using `total_bill` and `size`

Centre the predictor before modelling, `bill_c = total_bill - mean(total_bill)`, so that the intercept is the expected tip rate for an average bill.

In [ ]:
bill = tips["total_bill"].values
bill_c = bill - bill.mean()
size = tips["size"].values
tip_pct = tips["tip_pct"].values
print(f"Mean bill = ${bill.mean():.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bill, tip_pct, alpha=0.6, label="Bills")
ax.axhline(tip_pct.mean(), color="k", ls=":", label=f"Average tip rate ({tip_pct.mean():.1f}%)")
ax.set_title("Tip rate vs total bill")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip rate (% of bill)")
ax.legend()
plt.show()

## Simple linear regression

Model:
$
tip\_pct \sim \mathcal{N}(\alpha + \beta \cdot bill\_c,\ \sigma)
$

Store the mean as a deterministic variable, `μ = pm.Deterministic("μ", ...)`, because it is needed for the plots and for R² later.

In [ ]:
# α ~ N(16, 10): tip rate for an average bill, centred on a typical 16 %.
# β ~ N(0, 1): change in tip rate (pp) per $1 of bill; more than ±2 pp per $1 is implausible.
# σ ~ HalfNormal(10): scatter of tip rates around the line.
with pm.Model() as linear_model:
    α = pm.Normal("α", mu=16, sigma=10)
    β = pm.Normal("β", mu=0, sigma=1)
    σ = pm.HalfNormal("σ", sigma=10)
    μ = pm.Deterministic("μ", α + β * bill_c)
    y_obs = pm.Normal("y_obs", mu=μ, sigma=σ, observed=tip_pct)
    idata_linear = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_linear, "linear_model", exclude=("μ",))
# 1) Posterior summary of α, β, σ
display(az.summary(idata_linear, var_names=["~μ"]))
trace(idata_linear, ["~μ"], "Trace plots: tip rate ~ centred bill (Gaussian)")
beta_lin = idata_linear.posterior["β"].values.ravel()
print(f"P(β < 0) = {(beta_lin < 0).mean():.3f}")

# 2) Data, mean regression line and 94% HDI of μ
order_b = np.argsort(bill)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bill, tip_pct, alpha=0.5, label="Observed bills")
ax.plot(bill[order_b], idata_linear.posterior["μ"].mean(("chain", "draw")).values[order_b],
        color="k", lw=2, label="Posterior mean line")
az.plot_hdi(bill, idata_linear.posterior["μ"], hdi_prob=0.94, ax=ax,
            fill_kwargs={"alpha": 0.4, "label": "94% HDI of μ"})
ax.set_title("Simple linear regression: tip rate vs total bill")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip rate (% of bill)")
ax.legend()
plt.show()

# 3) Expected tip rate for a $10 and a $40 bill
post_lin = az.extract(idata_linear)
for b in [10, 40]:
    mu_b = (post_lin["α"] + post_lin["β"] * (b - bill.mean())).values
    lo, hi = az.hdi(mu_b, hdi_prob=0.94)
    print(f"Expected tip rate for a ${b} bill: {mu_b.mean():.2f}% (94% HDI {lo:.2f}% to {hi:.2f}%)")

The slope is clearly negative: β = −0.232 (94% HDI −0.309 to −0.151), and P(β < 0) = 1.000. For an average bill ($19.79), the expected tip rate is 16.08%. That drops from about 18.35% for a $10 bill to 11.39% for a $40 bill.

## Robust linear regression

Look at your scatter plot: a few small bills have unusually high tip rates. Replace the Gaussian likelihood with a **Student-t** likelihood so that these points have less influence:

$
tip\_pct \sim \text{StudentT}(\nu,\ \alpha + \beta \cdot bill\_c,\ \sigma)
$

In [ ]:
# Same priors as linear_model, plus ν for the tail heaviness.
with pm.Model() as robust_model:
    α = pm.Normal("α", mu=16, sigma=10)
    β = pm.Normal("β", mu=0, sigma=1)
    σ = pm.HalfNormal("σ", sigma=10)
    ν = pm.Exponential("ν", 1/30)
    μ = pm.Deterministic("μ", α + β * bill_c)
    y_obs = pm.StudentT("y_obs", nu=ν, mu=μ, sigma=σ, observed=tip_pct)
    idata_robust = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_robust, "robust_model", exclude=("μ",))
display(az.summary(idata_linear, var_names=["α", "β", "σ"]).assign(model="Gaussian"))
display(az.summary(idata_robust, var_names=["α", "β", "σ", "ν"]).assign(model="Student-t"))
trace(idata_robust, ["~μ"], "Trace plots: robust (Student-t) regression of tip rate")
beta_rob4 = idata_robust.posterior["β"].values.ravel()
print(f"P(β < 0): Gaussian = {(beta_lin < 0).mean():.3f}, Student-t = {(beta_rob4 < 0).mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bill, tip_pct, alpha=0.4, color="grey", label="Observed bills")
for idata_, name, col in [(idata_linear, "Gaussian", "C0"), (idata_robust, "Student-t", "C1")]:
    m = idata_.posterior["μ"].mean(("chain", "draw")).values
    ax.plot(bill[order_b], m[order_b], color=col, lw=2, label=f"{name} mean line")
    az.plot_hdi(bill, idata_.posterior["μ"], hdi_prob=0.94, ax=ax, color=col,
                fill_kwargs={"alpha": 0.25, "label": f"{name} 94% HDI"})
ax.set_title("Tip rate vs total bill: Gaussian vs Student-t fit")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip rate (% of bill)")
ax.legend()
plt.show()

The Student-t line is flatter (β = −0.182 vs −0.232). ν ≈ 5.0 shows heavy tails. The few small bills with very high tip rates (71%, 42%, 33%) were making the Gaussian line steeper at the left. The slope is still clearly negative, with P(β < 0) = 1.000 in both models.

## Polynomial regression

Add a quadratic term to check whether the relationship is curved:

$
\mu = \alpha + \beta_1 \cdot bill\_c + \beta_2 \cdot bill\_c^2
$

In [ ]:
# β2 ~ N(0, 0.1): bill_c² reaches ~900, so a curvature coefficient of 0.1 already bends the
# line by ~90 pp at the edges. This keeps the prior weak but on a sensible scale.
with pm.Model() as poly_model:
    α = pm.Normal("α", mu=16, sigma=10)
    β1 = pm.Normal("β1", mu=0, sigma=1)
    β2 = pm.Normal("β2", mu=0, sigma=0.1)
    σ = pm.HalfNormal("σ", sigma=10)
    μ = pm.Deterministic("μ", α + β1 * bill_c + β2 * bill_c**2)
    y_obs = pm.Normal("y_obs", mu=μ, sigma=σ, observed=tip_pct)
    idata_poly = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_poly, "poly_model", exclude=("μ",))
display(az.summary(idata_poly, var_names=["~μ"], round_to=4))
trace(idata_poly, ["~μ"], "Trace plots: quadratic regression of tip rate (Gaussian)")
b2_poly = idata_poly.posterior["β2"].values.ravel()
print(f"β2 94% HDI: {az.hdi(b2_poly, hdi_prob=0.94)},  P(β2 > 0) = {(b2_poly > 0).mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bill, tip_pct, alpha=0.5, label="Observed bills")
ax.plot(bill[order_b], idata_poly.posterior["μ"].mean(("chain", "draw")).values[order_b],
        color="k", lw=2, label="Quadratic mean curve")
az.plot_hdi(bill, idata_poly.posterior["μ"], hdi_prob=0.94, ax=ax,
            fill_kwargs={"alpha": 0.4, "label": "94% HDI of μ"})
ax.set_title("Quadratic regression (Gaussian): tip rate vs total bill")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip rate (% of bill)")
ax.legend()
plt.show()

In [ ]:
with pm.Model() as poly_robust_model:
    α = pm.Normal("α", mu=16, sigma=10)
    β1 = pm.Normal("β1", mu=0, sigma=1)
    β2 = pm.Normal("β2", mu=0, sigma=0.1)
    σ = pm.HalfNormal("σ", sigma=10)
    ν = pm.Exponential("ν", 1/30)
    μ = pm.Deterministic("μ", α + β1 * bill_c + β2 * bill_c**2)
    y_obs = pm.StudentT("y_obs", nu=ν, mu=μ, sigma=σ, observed=tip_pct)
    idata_poly_robust = pm.sample(**SAMPLE_KW)

diagnostics(idata_poly_robust, "poly_robust_model", exclude=("μ",))
display(az.summary(idata_poly_robust, var_names=["~μ"], round_to=4))
trace(idata_poly_robust, ["~μ"], "Trace plots: quadratic regression of tip rate (Student-t)")
b2_polyr = idata_poly_robust.posterior["β2"].values.ravel()
print(f"β2 94% HDI: {az.hdi(b2_polyr, hdi_prob=0.94)},  P(β2 > 0) = {(b2_polyr > 0).mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bill, tip_pct, alpha=0.4, color="grey", label="Observed bills")
for idata_, name, col in [(idata_poly, "Quadratic Gaussian", "C0"), (idata_poly_robust, "Quadratic Student-t", "C1")]:
    m = idata_.posterior["μ"].mean(("chain", "draw")).values
    ax.plot(bill[order_b], m[order_b], color=col, lw=2, label=f"{name} mean curve")
    az.plot_hdi(bill, idata_.posterior["μ"], hdi_prob=0.94, ax=ax, color=col,
                fill_kwargs={"alpha": 0.25, "label": f"{name} 94% HDI"})
ax.set_title("Quadratic fits of tip rate: Gaussian vs Student-t")
ax.set_xlabel("Total bill ($)")
ax.set_ylabel("Tip rate (% of bill)")
ax.legend()
plt.show()

With a Gaussian likelihood, β2 = 0.0089 (HDI 0.0029 to 0.0151), so the curve looks real. Once the likelihood is Student-t, β2 drops to 0.0028 and its HDI (−0.0026 to 0.0085) includes 0. The fitted curves in the plot also show that the Gaussian curve bends up at both ends, while the robust one is close to a straight line.

## Multivariable regression

Use the bill amount and the party size together:

$
\mu = \alpha + \beta_1 \cdot bill\_c + \beta_2 \cdot size
$

In [ ]:
# β2 ~ N(0, 2): one extra person changing the tip rate by more than ~4 pp would be surprising.
# size is not centred (as in the formula), so α is the tip rate for an average bill and size = 0;
# the wider N(16, 15) prior on α allows for that.
with pm.Model() as multi_model:
    α = pm.Normal("α", mu=16, sigma=15)
    β1 = pm.Normal("β1", mu=0, sigma=1)
    β2 = pm.Normal("β2", mu=0, sigma=2)
    σ = pm.HalfNormal("σ", sigma=10)
    μ = pm.Deterministic("μ", α + β1 * bill_c + β2 * size)
    y_obs = pm.Normal("y_obs", mu=μ, sigma=σ, observed=tip_pct)
    idata_multi = pm.sample(**SAMPLE_KW)

In [ ]:
diagnostics(idata_multi, "multi_model", exclude=("μ",))
display(az.summary(idata_multi, var_names=["~μ"]))
trace(idata_multi, ["~μ"], "Trace plots: tip rate ~ centred bill + party size")
b1_m = idata_multi.posterior["β1"].values.ravel()
b2_m = idata_multi.posterior["β2"].values.ravel()
print(f"β1: mean {b1_m.mean():.3f}, 94% HDI {az.hdi(b1_m)}, P(β1 < 0) = {(b1_m < 0).mean():.3f}")
print(f"β2: mean {b2_m.mean():.3f}, 94% HDI {az.hdi(b2_m)}, P(β2 > 0) = {(b2_m > 0).mean():.3f}")

fig, ax = plt.subplots(figsize=(7, 3))
az.plot_forest([idata_linear, idata_multi], model_names=["linear_model", "multi_model"],
               var_names=["β", "β1", "β2"], combined=True, ax=ax)
ax.set_title("Slopes of tip rate: simple vs multivariable model (94% HDI)")
ax.set_xlabel("Change in tip rate (pp)")
plt.show()

Once size is added, the bill slope gets a bit steeper (β1 = −0.267, HDI −0.363 to −0.172). β2 = 0.552 pp per extra person, but its HDI (−0.333 to 1.392) includes 0 and P(β2 > 0) is only 0.881.

## Model checking

For the Gaussian linear, robust linear, polynomial and multivariable models:

1) run a posterior predictive check, and
2) compute the Bayesian R² with `az.r2_score`.

For R², pass the posterior draws of the **mean** `μ` (one row per draw, one column per observation), not the posterior predictive draws. Predictive draws also contain the observation noise, which inflates R² even for a model that explains nothing.

In [ ]:
models = {
    "linear_model": (linear_model, idata_linear),
    "robust_model": (robust_model, idata_robust),
    "poly_model": (poly_model, idata_poly),
    "multi_model": (multi_model, idata_multi),
}

# 1) Posterior predictive checks
ppcs = {}
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
for ax, (name, (model, idata_)) in zip(axes.ravel(), models.items()):
    with model:
        ppcs[name] = pm.sample_posterior_predictive(idata_, random_seed=42, progressbar=False)
    ppc_plot(ppcs[name], f"PPC: {name}", "Tip rate (% of bill)", ax=ax)
    ax.set_xlim(-20, 80)
plt.tight_layout()
plt.show()

# 2) Bayesian R² from the posterior draws of μ (not the predictive draws)
r2 = {}
for name, (model, idata_) in models.items():
    mu_draws = idata_.posterior["μ"].values.reshape(-1, len(tip_pct))
    r2[name] = az.r2_score(tip_pct, mu_draws)
r2_table = pd.DataFrame(r2).T
r2_table.columns = ["R² (mean)", "R² (sd)"]
r2_table.round(3)

**Model checks:** The three Gaussian models produce wider and flatter predictive densities than the data and miss the sharp peak around 15%. They also predict some negative tip rates. The robust model fits the peak much better and has heavy tails that can reach the extreme values. Its R² is the lowest (0.075). The predictive density and Bayesian R² assess different aspects of fit. A lower R² does not by itself show that the predictive distribution is worse. All saved R² values are small.

## Short reflection (1–2 sentences per question on PART 3)
1. Interpret the slope `β` of the simple linear model in words.
2. What changed between the Gaussian and the Student-t regression, and why?
3. Is the curvature found by the polynomial model real? Use both quadratic fits to justify your answer.
4. Part 2 compared the tip rate across party sizes. What does `β2` of the multivariable model tell you about party size once the bill amount is taken into account?
5. The slope is credibly different from zero, yet R² is small. What does that tell the restaurant owner?

**Reflection**

1. **Slope β:** for every extra $1 on the bill, the expected tip rate falls by about 0.23 percentage points (94% HDI 0.15 to 0.31). That works out to about 18.4% for a $10 bill and 11.4% for a $40 bill.
2. **Gaussian vs Student-t:** the robust slope is flatter (−0.182 vs −0.232) and σ drops from 5.8 to 3.9. A few small bills with very high tip rates (up to 71%) have less pull under the heavy-tailed Student-t (ν ≈ 5). The negative direction stays the same.
3. **Curvature:** probably not real. The Gaussian quadratic gives β2 = 0.0089 with an HDI above 0. The Student-t quadratic gives β2 = 0.0028, and its HDI (−0.0026 to 0.0085) includes 0. So the curve mostly comes from the same few extreme small bills.
4. **Party size (β2):** with the bill held fixed, one extra person goes with about +0.55 pp in tip rate, but the HDI (−0.33 to 1.39) includes 0. The lower raw tip rates for large parties in Part 3 mostly reflect their bigger bills, and once the bill is accounted for there is no clear size effect.
5. **Credible slope, small R²:** the bill size is reliably linked to a lower tip rate, but it only explains about 12% of the variation (R² ≈ 0.12). For the owner, this is useful for average planning (bigger bills → lower percentage). It is not useful for predicting what one particular table will tip.